# Phase 3 — CT-BNet (Causal Temporal Bipartite Network) + ablations A–E

Implements the architecture spec on top of Phase 1 / Phase 1c features. Runs five ablations end-to-end on the same temporal split and reports per-timestep breakdown across all of them.

## Five ablations

| | Description | Trained model |
|---|---|---|
| **A** | raw 182 per-tx features | RandomForest |
| **B** | A + 103 Phase-1 trajectory features | RandomForest |
| **C** | B + 7 Phase-1c bipartite-structural features | RandomForest |
| **D** | GRU wallet memory + bipartite attention (no adversarial) | CT-BNet, `λ_adv = 0` |
| **E** | Full CT-BNet (D + drift-adversarial) | CT-BNet, `λ_adv = 0.1` |

Comparisons of interest:
- **B vs A**: Phase-1 trajectory signal (already established at +0.005 F1 / +0.017 PR-AUC)
- **C vs B**: bipartite structural signal beyond trajectory aggregates
- **D vs C**: learned temporal-graph representations vs. hand-engineered ones
- **E vs D**: drift-adversarial regularisation impact at the t=43 cliff

## Causality contract

For tx `T` at time `t`:
- **Phase-1 traj features**: strict `τ < t` priors (unchanged from phase 1).
- **Phase-1c L1 features**: PageRank on the cumulative bipartite graph with edges where `tx_endpoint.t' ≤ t`. PageRank uses no labels.
- **CT-BNet wallet memory**: chronological processing. At timestep `t`, all events from `t' ≤ t` have been incorporated into wallet memories before any classification at `t > t` happens. Test-set tx labels are never used in memory updates.
- **No `wallets_features.txt`** (lifetime aggregates leak), **no AddrAddr edges** (no timestamps).

## CT-BNet architecture (per the spec, with one mild relaxation)

1. **Wallet memory** (Module 1): `m_w ∈ ℝ^{D_MEM}` per wallet, init zero. Updated each timestep by GRU on aggregated event embeddings.
2. **Bipartite attention** (Module 2): for target tx `T` at `t`, query `[x_T ‖ φ(t)]` attends over incident wallets' memory keys `[m_w ‖ φ(Δt) ‖ seg]`.
3. **Drift adversarial** (Module 3): discriminator predicts normalised timestep from `new_mem`; gradient reversal makes the GRU produce timestep-invariant features.
4. **Fusion classifier** (Module 4): `[x_T ‖ ctx ‖ x_T - W_r·ctx]` → MLP → 2 logits.

**Mild relaxation from spec**: rather than reading `m_w(t⁻)` for classification (which would require TBPTT to give the GRU any gradient and blow CPU memory), we aggregate all events at `t` per wallet, run the GRU update, then classify with the post-update memory. T's own event contributes `1/n` of the aggregated update where `n` is the number of events for that wallet at `t`. This is still causal (`τ ≤ t`) and lets the GRU train end-to-end with K=1 windows on CPU.

## Data split — identical to Phase 1 / 1c

- Train: labelled txs with `t ≤ 34`. Test: labelled txs with `t ≥ 35`.

Wall clock target: ~10 min total (~1 min for traj + L1 features, ~30s for ablations A/B/C, ~3 min × 2 for D and E).


In [1]:
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import scipy.sparse as sp
from pathlib import Path
from collections import defaultdict
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    f1_score, roc_auc_score, average_precision_score,
    classification_report, precision_recall_curve,
)

ROOT = Path.cwd()
while not (ROOT / "actors_data").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
WALLETS_DIR      = ROOT / "actors_data"
TRANSACTIONS_DIR = ROOT / "transactions_data"
assert WALLETS_DIR.exists() and TRANSACTIONS_DIR.exists()
print(f"repo root: {ROOT}")

# Same data split as Phase 1 / 1c
TRAIN_END    = 34
N_TIME_STEPS = 49
RANDOM_SEED  = 175
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

# Phase 1 trajectory engineering caps (unchanged)
MAX_INCIDENT_PER_SIDE = 32
MAX_CO_WALLETS        = 150
RECENCY_SENTINEL      = N_TIME_STEPS * 2
DECAY_RATE            = 0.2

# Phase 1c PageRank settings
PR_DAMP        = 0.85
PR_ITERATIONS  = 30

# CT-BNet hyperparameters
D_EVENT      = 64       # event embedding dim (output of tx_proj + time encoding)
D_MEM        = 64       # GRU hidden / wallet memory dim
D_ATTN       = 64       # attention key/query/value dim
D_SEG        = 8        # segment embedding dim (input vs output)
W_PER_SIDE   = 4        # max incident wallets per side for attention
W_TOTAL      = 2 * W_PER_SIDE
DROPOUT      = 0.3
LR           = 1e-3
WEIGHT_DECAY = 5e-4
EPOCHS       = 8
CLASS_WEIGHT = 3.0      # less aggressive than balanced ~9.2 (see Phase 2 / 2-simplified)
LAMBDA_ADV_E = 0.1      # for ablation E (full CT-BNet)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {DEVICE}")


repo root: /Users/jarayliu/Documents/GitHub/stat175-final
device: cpu


In [2]:
print("Loading transactions...")
tx_features_df = pd.read_csv(TRANSACTIONS_DIR / "txs_features.csv")
tx_classes_df  = pd.read_csv(TRANSACTIONS_DIR / "txs_classes.csv")
tx_classes_df["label"] = tx_classes_df["class"].map({1: 1, 2: 0, 3: -1}).astype(np.int8)
tx_id_to_idx = {tid: i for i, tid in enumerate(tx_features_df["txId"].values)}
N_tx = len(tx_features_df)
tx_feat_cols = [c for c in tx_features_df.columns if c not in ("txId", "Time step")]
F_TX = len(tx_feat_cols)
merged = tx_features_df[["txId","Time step"]].merge(tx_classes_df[["txId","label"]], on="txId", how="left")
tx_time  = merged["Time step"].astype(np.int64).values
tx_label = merged["label"].fillna(-1).astype(np.int64).values
tx_X_raw = tx_features_df[tx_feat_cols].fillna(0).astype(np.float32).values
print(f"  N_tx={N_tx:,}  F_TX={F_TX}")
print(f"  labels: illicit={int((tx_label==1).sum()):,}  licit={int((tx_label==0).sum()):,}  unknown={int((tx_label==-1).sum()):,}")

print("\nLoading actor bipartite edges...")
addr_tx_df = pd.read_csv(WALLETS_DIR / "AddrTx_edgelist.txt")
tx_addr_df = pd.read_csv(WALLETS_DIR / "TxAddr_edgelist.txt")
wallet_addrs = sorted(set(addr_tx_df["input_address"].unique()) | set(tx_addr_df["output_address"].unique()))
wallet_addr_to_idx = {a: i for i, a in enumerate(wallet_addrs)}
N_w = len(wallet_addr_to_idx)
print(f"  unique wallets: {N_w:,}")

print("\nBuilding per-wallet timelines...")
def edges_with_time(edge_df, addr_col, tx_col):
    df = edge_df[[addr_col, tx_col]].copy()
    df["w"]      = df[addr_col].map(wallet_addr_to_idx)
    df["tx"]     = df[tx_col].map(tx_id_to_idx)
    df = df.dropna(subset=["w", "tx"])
    df["w"]      = df["w"].astype(np.int64)
    df["tx"]     = df["tx"].astype(np.int64)
    df["t"]      = tx_time[df["tx"].values]
    df["tx_lab"] = tx_label[df["tx"].values]
    return df[["w","tx","t","tx_lab"]]

at = edges_with_time(addr_tx_df, "input_address",  "txId")
ta = edges_with_time(tx_addr_df, "output_address", "txId")
incident_in  = defaultdict(list)
incident_out = defaultdict(list)
for tx, w in zip(at["tx"].values,  at["w"].values):  incident_in[int(tx)].append(int(w))
for tx, w in zip(ta["tx"].values,  ta["w"].values):  incident_out[int(tx)].append(int(w))

all_edges = pd.concat([at, ta], ignore_index=True)
all_edges = all_edges.drop_duplicates(subset=["w", "tx"]).sort_values(["w", "t"]).reset_index(drop=True)
g = all_edges.groupby("w", sort=False)
wallet_t_arr   = {}
wallet_tx_arr  = {}
wallet_lab_arr = {}
for w, sub in g:
    wallet_t_arr[int(w)]   = sub["t"].values.astype(np.int64)
    wallet_tx_arr[int(w)]  = sub["tx"].values.astype(np.int64)
    wallet_lab_arr[int(w)] = sub["tx_lab"].values.astype(np.int64)

wallet_has_illicit_by = {}
for w, labs in wallet_lab_arr.items():
    illicit_mask = (labs == 1)
    if illicit_mask.any():
        wallet_has_illicit_by[w] = int(wallet_t_arr[w][illicit_mask].min())

wallet_event_count = {w: len(tarr) for w, tarr in wallet_t_arr.items()}
print(f"  wallets with timelines: {len(wallet_t_arr):,}")
print(f"  total wallet-tx pairs: {len(all_edges):,}")


Loading transactions...
  N_tx=203,769  F_TX=182
  labels: illicit=4,545  licit=42,019  unknown=157,205

Loading actor bipartite edges...
  unique wallets: 822,942

Building per-wallet timelines...
  wallets with timelines: 822,942
  total wallet-tx pairs: 1,268,260


In [3]:
# Phase 1 trajectory features — exact port from phase1c.
agg_feat_names = ["total_BTC", "fees", "num_input_addresses", "num_output_addresses"]
agg_feat_idxs  = [tx_feat_cols.index(c) for c in agg_feat_names]
total_btc_idx  = tx_feat_cols.index("total_BTC")
F_agg          = len(agg_feat_names)

def pick_top_wallets(wlist, k=MAX_INCIDENT_PER_SIDE):
    if len(wlist) <= k: return list(wlist)
    cnts  = np.array([wallet_event_count.get(w, 0) for w in wlist], dtype=np.int64)
    order = np.argsort(-cnts, kind="stable")
    return [wlist[i] for i in order[:k]]

def per_wallet_priors(w, t_query):
    tarr = wallet_t_arr.get(w)
    if tarr is None: return None
    cut = int(np.searchsorted(tarr, t_query, side="left"))
    if cut == 0: return None
    prior_t   = tarr[:cut]
    prior_lab = wallet_lab_arr[w][:cut]
    prior_tx  = wallet_tx_arr[w][:cut]
    illicit_mask = (prior_lab == 1)
    n_illicit = int(illicit_mask.sum())
    n_licit   = int((prior_lab == 0).sum())
    last_illicit_t = int(prior_t[illicit_mask].max()) if n_illicit > 0 else -1
    if n_illicit > 0:
        decay_w = np.exp(-DECAY_RATE * (t_query - prior_t[illicit_mask]).astype(np.float64))
        decayed_illicit_score = float(decay_w.sum())
    else:
        decayed_illicit_score = 0.0
    illicit_frac = n_illicit / max(n_illicit + n_licit, 1)
    co_wallets = set()
    for tx_i in prior_tx:
        tx_i_int = int(tx_i)
        for co_w in incident_in.get(tx_i_int,  []):
            if co_w != w: co_wallets.add(co_w)
        for co_w in incident_out.get(tx_i_int, []):
            if co_w != w: co_wallets.add(co_w)
        if len(co_wallets) >= MAX_CO_WALLETS: break
    n_co_wallets = len(co_wallets)
    n_co_illicit = sum(1 for cw in co_wallets
                       if wallet_has_illicit_by.get(cw, N_TIME_STEPS + 1) < t_query)
    unique_in_partners, unique_out_partners = set(), set()
    for tx_i in prior_tx:
        tx_i_int = int(tx_i)
        for co_w in incident_in.get(tx_i_int,  []):
            if co_w != w: unique_in_partners.add(co_w)
        for co_w in incident_out.get(tx_i_int, []):
            if co_w != w: unique_out_partners.add(co_w)
    n_in_partners  = len(unique_in_partners)
    n_out_partners = len(unique_out_partners)
    fan_asymmetry  = n_out_partners / max(n_in_partners, 1)
    age      = int(t_query - prior_t[0])
    recency  = int(t_query - prior_t[-1])
    n_recent_5 = int(((t_query - prior_t) <= 5).sum())
    n_recent_1 = int(((t_query - prior_t) <= 1).sum())
    if cut >= 2:
        iat = np.diff(prior_t.astype(np.float64))
        iat_mean = float(iat.mean()); iat_std = float(iat.std())
        burstiness = float((iat_std - iat_mean) / (iat_std + iat_mean + 1e-8))
    else:
        iat_mean, iat_std, burstiness = 0.0, 0.0, 0.0
    velocity = n_recent_5 / max(cut, 1)
    feat_vals = tx_X_raw[prior_tx][:, agg_feat_idxs]
    return {
        "n": int(cut), "n_illicit": n_illicit, "n_licit": n_licit,
        "illicit_frac": illicit_frac, "last_illicit_t": last_illicit_t,
        "decayed_illicit": decayed_illicit_score,
        "n_co_wallets": n_co_wallets, "n_co_illicit": n_co_illicit,
        "co_illicit_frac": n_co_illicit / max(n_co_wallets, 1),
        "n_in_partners": n_in_partners, "n_out_partners": n_out_partners,
        "fan_asymmetry": fan_asymmetry,
        "first_seen_t": int(prior_t[0]), "last_seen_t": int(prior_t[-1]),
        "age": age, "recency": recency,
        "n_recent_5": n_recent_5, "n_recent_1": n_recent_1,
        "iat_mean": iat_mean, "iat_std": iat_std,
        "burstiness": burstiness, "velocity": velocity,
        "feat_means": feat_vals.mean(axis=0), "feat_maxes": feat_vals.max(axis=0),
    }

def aggregate_side(summaries, side, t_T):
    n_total = len(summaries)
    valid   = [s for s in summaries if s is not None]
    n_w_prior = len(valid)
    out = {}; p = side
    out[f"{p}_n_wallets"]              = n_total
    out[f"{p}_n_wallets_with_prior"]   = n_w_prior
    out[f"{p}_frac_first_appearance"]  = (n_total - n_w_prior) / max(n_total, 1)
    if not valid:
        # Match phase 1's exact key insertion order so traj_X column order is identical.
        out[f"{p}_n_priors_sum"]                = 0.0
        out[f"{p}_n_priors_max"]                = 0.0
        out[f"{p}_n_illicit_sum"]               = 0.0
        out[f"{p}_n_illicit_max"]               = 0.0
        out[f"{p}_n_licit_sum"]                 = 0.0
        out[f"{p}_n_wallets_with_illicit"]      = 0.0
        out[f"{p}_n_wallets_illicit_frac_gt0"]  = 0.0
        out[f"{p}_n_wallets_illicit_frac_gt50"] = 0.0
        out[f"{p}_illicit_frac_max"]            = 0.0
        out[f"{p}_illicit_frac_mean"]           = 0.0
        out[f"{p}_decayed_illicit_max"]         = 0.0
        out[f"{p}_decayed_illicit_sum"]         = 0.0
        out[f"{p}_recency_to_illicit_min"]      = float(RECENCY_SENTINEL)
        out[f"{p}_co_illicit_sum"]              = 0.0
        out[f"{p}_co_illicit_max"]              = 0.0
        out[f"{p}_co_illicit_frac_max"]         = 0.0
        out[f"{p}_co_illicit_frac_mean"]        = 0.0
        out[f"{p}_n_co_wallets_sum"]            = 0.0
        out[f"{p}_fan_asymmetry_max"]           = 0.0
        out[f"{p}_fan_asymmetry_mean"]          = 0.0
        out[f"{p}_n_in_partners_max"]           = 0.0
        out[f"{p}_n_out_partners_max"]          = 0.0
        out[f"{p}_frac_single_use"]             = 0.0
        out[f"{p}_age_max"]                     = 0.0
        out[f"{p}_age_mean"]                    = 0.0
        out[f"{p}_recency_min"]                 = float(RECENCY_SENTINEL)
        out[f"{p}_n_recent_5_sum"]              = 0.0
        out[f"{p}_n_recent_5_max"]              = 0.0
        out[f"{p}_n_recent_1_sum"]              = 0.0
        out[f"{p}_velocity_max"]                = 0.0
        out[f"{p}_velocity_mean"]               = 0.0
        out[f"{p}_burstiness_max"]              = 0.0
        out[f"{p}_burstiness_mean"]             = 0.0
        out[f"{p}_iat_mean_min"]                = 0.0
        out[f"{p}_iat_std_max"]                 = 0.0
        for nm in agg_feat_names:
            out[f"{p}_prior_{nm}_mean_max"] = 0.0
            out[f"{p}_prior_{nm}_max_max"]  = 0.0
        return out
    ns        = np.array([s["n"]              for s in valid], dtype=np.float64)
    nis       = np.array([s["n_illicit"]      for s in valid], dtype=np.float64)
    nls       = np.array([s["n_licit"]        for s in valid], dtype=np.float64)
    ill_frac  = np.array([s["illicit_frac"]   for s in valid], dtype=np.float64)
    dec_ill   = np.array([s["decayed_illicit"]for s in valid], dtype=np.float64)
    last_ill  = np.array([s["last_illicit_t"] for s in valid], dtype=np.int64)
    has_ill   = (last_ill >= 0)
    rec_ill   = np.where(has_ill, t_T - last_ill, RECENCY_SENTINEL).astype(np.float64)
    co_ill   = np.array([s["n_co_illicit"]    for s in valid], dtype=np.float64)
    co_n     = np.array([s["n_co_wallets"]    for s in valid], dtype=np.float64)
    co_frac  = np.array([s["co_illicit_frac"] for s in valid], dtype=np.float64)
    fan_a    = np.array([s["fan_asymmetry"]   for s in valid], dtype=np.float64)
    n_inp    = np.array([s["n_in_partners"]   for s in valid], dtype=np.float64)
    n_outp   = np.array([s["n_out_partners"]  for s in valid], dtype=np.float64)
    ages = np.array([s["age"]      for s in valid], dtype=np.float64)
    recs = np.array([s["recency"]  for s in valid], dtype=np.float64)
    nr5  = np.array([s["n_recent_5"] for s in valid], dtype=np.float64)
    nr1  = np.array([s["n_recent_1"] for s in valid], dtype=np.float64)
    vel  = np.array([s["velocity"] for s in valid], dtype=np.float64)
    bur  = np.array([s["burstiness"] for s in valid], dtype=np.float64)
    iam  = np.array([s["iat_mean"] for s in valid], dtype=np.float64)
    ias  = np.array([s["iat_std"]  for s in valid], dtype=np.float64)
    feat_means = np.stack([s["feat_means"] for s in valid], axis=0)
    feat_maxes = np.stack([s["feat_maxes"] for s in valid], axis=0)
    out[f"{p}_n_priors_sum"] = float(ns.sum()); out[f"{p}_n_priors_max"] = float(ns.max())
    out[f"{p}_n_illicit_sum"] = float(nis.sum()); out[f"{p}_n_illicit_max"] = float(nis.max())
    out[f"{p}_n_licit_sum"] = float(nls.sum())
    out[f"{p}_n_wallets_with_illicit"] = float(has_ill.sum())
    out[f"{p}_n_wallets_illicit_frac_gt0"]  = float((ill_frac > 0.0).sum())
    out[f"{p}_n_wallets_illicit_frac_gt50"] = float((ill_frac > 0.5).sum())
    out[f"{p}_illicit_frac_max"] = float(ill_frac.max())
    out[f"{p}_illicit_frac_mean"] = float(ill_frac.mean())
    out[f"{p}_decayed_illicit_max"] = float(dec_ill.max())
    out[f"{p}_decayed_illicit_sum"] = float(dec_ill.sum())
    out[f"{p}_recency_to_illicit_min"] = float(rec_ill.min())
    out[f"{p}_co_illicit_sum"] = float(co_ill.sum()); out[f"{p}_co_illicit_max"] = float(co_ill.max())
    out[f"{p}_co_illicit_frac_max"] = float(co_frac.max())
    out[f"{p}_co_illicit_frac_mean"] = float(co_frac.mean())
    out[f"{p}_n_co_wallets_sum"] = float(co_n.sum())
    out[f"{p}_fan_asymmetry_max"] = float(fan_a.max())
    out[f"{p}_fan_asymmetry_mean"] = float(fan_a.mean())
    out[f"{p}_n_in_partners_max"] = float(n_inp.max())
    out[f"{p}_n_out_partners_max"] = float(n_outp.max())
    out[f"{p}_frac_single_use"] = sum(1 for s in valid if s["n"] == 1) / max(n_w_prior, 1)
    out[f"{p}_age_max"] = float(ages.max()); out[f"{p}_age_mean"] = float(ages.mean())
    out[f"{p}_recency_min"] = float(recs.min())
    out[f"{p}_n_recent_5_sum"] = float(nr5.sum()); out[f"{p}_n_recent_5_max"] = float(nr5.max())
    out[f"{p}_n_recent_1_sum"] = float(nr1.sum())
    out[f"{p}_velocity_max"] = float(vel.max()); out[f"{p}_velocity_mean"] = float(vel.mean())
    out[f"{p}_burstiness_max"] = float(bur.max()); out[f"{p}_burstiness_mean"] = float(bur.mean())
    out[f"{p}_iat_mean_min"] = float(iam.min()); out[f"{p}_iat_std_max"] = float(ias.max())
    for k, nm in enumerate(agg_feat_names):
        out[f"{p}_prior_{nm}_mean_max"] = float(feat_means[:, k].max())
        out[f"{p}_prior_{nm}_max_max"]  = float(feat_maxes[:, k].max())
    return out

print("Computing trajectory features for all txs (~30s)...")
t0 = time.time()
traj_rows = []
for tx_idx in range(N_tx):
    t_T = int(tx_time[tx_idx])
    in_w  = pick_top_wallets(incident_in.get(tx_idx,  []))
    out_w = pick_top_wallets(incident_out.get(tx_idx, []))
    in_summ  = [per_wallet_priors(w, t_T) for w in in_w]
    out_summ = [per_wallet_priors(w, t_T) for w in out_w]
    row = {}
    row.update(aggregate_side(in_summ,  "in",  t_T))
    row.update(aggregate_side(out_summ, "out", t_T))
    row["both_sides_have_illicit"] = float(
        row["in_n_wallets_with_illicit"] > 0 and row["out_n_wallets_with_illicit"] > 0)
    row["total_n_illicit_priors"]  = row["in_n_illicit_sum"] + row["out_n_illicit_sum"]
    row["total_n_wallets_with_illicit"] = row["in_n_wallets_with_illicit"] + row["out_n_wallets_with_illicit"]
    row["total_co_illicit"]        = row["in_co_illicit_sum"] + row["out_co_illicit_sum"]
    row["min_recency_to_illicit"]  = min(row["in_recency_to_illicit_min"], row["out_recency_to_illicit_min"])
    row["max_illicit_frac_either_side"] = max(row["in_illicit_frac_max"], row["out_illicit_frac_max"])
    row["max_decayed_illicit_either"]   = max(row["in_decayed_illicit_max"], row["out_decayed_illicit_max"])
    row["max_co_illicit_frac_either"]   = max(row["in_co_illicit_frac_max"], row["out_co_illicit_frac_max"])
    row["total_frac_first_appearance"]  = (
        (row["in_frac_first_appearance"] * max(row["in_n_wallets"], 1) +
         row["out_frac_first_appearance"] * max(row["out_n_wallets"], 1))
        / max(row["in_n_wallets"] + row["out_n_wallets"], 1))
    T_total_btc = float(tx_X_raw[tx_idx, total_btc_idx])
    max_prior_btc  = max(row["in_prior_total_BTC_max_max"],  row["out_prior_total_BTC_max_max"])
    mean_prior_btc = max(row["in_prior_total_BTC_mean_max"], row["out_prior_total_BTC_mean_max"])
    row["T_btc_vs_max_prior"]  = T_total_btc / max(max_prior_btc, 1.0)
    row["T_btc_vs_mean_prior"] = T_total_btc / max(mean_prior_btc, 1.0)
    traj_rows.append(row)
    if tx_idx > 0 and tx_idx % 50_000 == 0:
        print(f"  tx {tx_idx:>7,}/{N_tx:,}  ({time.time()-t0:.0f}s elapsed)")

traj_df = pd.DataFrame(traj_rows)
traj_X  = traj_df.values.astype(np.float32)
traj_cols = list(traj_df.columns)
F_TRAJ  = traj_X.shape[1]
print(f"  done: traj_X={traj_X.shape} ({time.time()-t0:.1f}s)")


Computing trajectory features for all txs (~30s)...
  tx  50,000/203,769  (5s elapsed)
  tx 100,000/203,769  (14s elapsed)
  tx 150,000/203,769  (21s elapsed)
  tx 200,000/203,769  (27s elapsed)
  done: traj_X=(203769, 103) (31.1s)


In [4]:
# Phase 1c Layer-1 bipartite-structural features — exact port.
print("Computing Layer 1 bipartite structural features...")
N_total = N_w + N_tx
W_edges = all_edges["w"].values.astype(np.int64)
T_edges = all_edges["tx"].values.astype(np.int64)
T_times = all_edges["t"].values.astype(np.int64)
edge_order = np.argsort(T_times)
W_edges_sorted = W_edges[edge_order]
T_edges_sorted = T_edges[edge_order]
T_times_sorted = T_times[edge_order]
incident_in_arr  = {tx: list(set(ws)) for tx, ws in incident_in.items()}
incident_out_arr = {tx: list(set(ws)) for tx, ws in incident_out.items()}

bp_pagerank        = np.zeros(N_tx, dtype=np.float32)
bp_in_degree       = np.zeros(N_tx, dtype=np.float32)
bp_out_degree      = np.zeros(N_tx, dtype=np.float32)
bp_max_nbr_pr      = np.zeros(N_tx, dtype=np.float32)
bp_mean_nbr_pr     = np.zeros(N_tx, dtype=np.float32)
bp_max_in_nbr_pr   = np.zeros(N_tx, dtype=np.float32)
bp_max_out_nbr_pr  = np.zeros(N_tx, dtype=np.float32)

t0 = time.time()
for t in range(1, N_TIME_STEPS + 1):
    cut = int(np.searchsorted(T_times_sorted, t, side="right"))
    if cut == 0: continue
    we = W_edges_sorted[:cut]
    te = T_edges_sorted[:cut]
    rows = np.concatenate([we, N_w + te])
    cols = np.concatenate([N_w + te, we])
    data = np.ones(rows.shape[0], dtype=np.float32)
    A = sp.csr_matrix((data, (rows, cols)), shape=(N_total, N_total))
    deg = np.asarray(A.sum(axis=1)).flatten()
    inv_deg = np.zeros_like(deg)
    nz = deg > 0
    inv_deg[nz] = 1.0 / deg[nz]
    teleport = (1.0 - PR_DAMP) / N_total
    pr = np.full(N_total, 1.0 / N_total, dtype=np.float32)
    for _ in range(PR_ITERATIONS):
        weighted = pr * inv_deg
        pr = teleport + PR_DAMP * (A @ weighted).astype(np.float32)
    txs_at_t = np.where(tx_time == t)[0]
    for tx_idx in txs_at_t:
        in_w  = incident_in_arr.get(int(tx_idx),  [])
        out_w = incident_out_arr.get(int(tx_idx), [])
        all_nbr = list(set(in_w) | set(out_w))
        bp_pagerank[tx_idx]  = pr[N_w + tx_idx]
        bp_in_degree[tx_idx]  = float(len(in_w))
        bp_out_degree[tx_idx] = float(len(out_w))
        if all_nbr:
            nbr_pr = pr[all_nbr]
            bp_max_nbr_pr[tx_idx]  = float(nbr_pr.max())
            bp_mean_nbr_pr[tx_idx] = float(nbr_pr.mean())
        if in_w:
            bp_max_in_nbr_pr[tx_idx]  = float(pr[in_w].max())
        if out_w:
            bp_max_out_nbr_pr[tx_idx] = float(pr[out_w].max())
    if t % 10 == 0 or t == N_TIME_STEPS:
        print(f"  t={t:2d}  edges_active={cut:>9,}  ({time.time()-t0:.1f}s elapsed)")

L1_X = np.column_stack([
    bp_pagerank, bp_in_degree, bp_out_degree,
    bp_max_nbr_pr, bp_mean_nbr_pr, bp_max_in_nbr_pr, bp_max_out_nbr_pr,
]).astype(np.float32)
L1_cols = ["bp_pagerank","bp_in_degree","bp_out_degree","bp_max_neighbor_pr",
           "bp_mean_neighbor_pr","bp_max_input_neighbor_pr","bp_max_output_neighbor_pr"]
F_L1 = L1_X.shape[1]
print(f"  done: L1_X={L1_X.shape} ({time.time()-t0:.1f}s)")


Computing Layer 1 bipartite structural features...
  t=10  edges_active=  349,941  (1.5s elapsed)
  t=20  edges_active=  546,600  (3.5s elapsed)
  t=30  edges_active=  767,134  (6.2s elapsed)
  t=40  edges_active=1,012,737  (9.0s elapsed)
  t=49  edges_active=1,268,260  (11.6s elapsed)
  done: L1_X=(203769, 7) (11.6s)


In [5]:
# === Chronological data structures for CT-BNet ===
print("Building chronological data structures...")

# txs_at_t[t]: numpy int64 array of tx indices active at exactly t
txs_at_t = [np.empty(0, dtype=np.int64) for _ in range(N_TIME_STEPS + 1)]
for t in range(1, N_TIME_STEPS + 1):
    txs_at_t[t] = np.where(tx_time == t)[0].astype(np.int64)

# parts_at_t[t]: list of (wallet_idx, tx_idx, segment) for participations at t
parts_w_at_t   = [None] * (N_TIME_STEPS + 1)
parts_tx_at_t  = [None] * (N_TIME_STEPS + 1)
parts_seg_at_t = [None] * (N_TIME_STEPS + 1)
for t in range(1, N_TIME_STEPS + 1):
    txs_t = set(int(x) for x in txs_at_t[t])
    if not txs_t:
        parts_w_at_t[t]   = np.empty(0, dtype=np.int64)
        parts_tx_at_t[t]  = np.empty(0, dtype=np.int64)
        parts_seg_at_t[t] = np.empty(0, dtype=np.int64)
        continue
    rows = []
    for tx_idx in txs_t:
        seen = set()
        for w in incident_in.get(tx_idx, []):
            if (w, tx_idx) in seen: continue
            seen.add((w, tx_idx))
            rows.append((w, tx_idx, 0))   # 0 = INPUT
        for w in incident_out.get(tx_idx, []):
            if (w, tx_idx) in seen: continue
            seen.add((w, tx_idx))
            rows.append((w, tx_idx, 1))   # 1 = OUTPUT
    if rows:
        ws, txs, segs = zip(*rows)
        parts_w_at_t[t]   = np.array(ws,   dtype=np.int64)
        parts_tx_at_t[t]  = np.array(txs,  dtype=np.int64)
        parts_seg_at_t[t] = np.array(segs, dtype=np.int64)
    else:
        parts_w_at_t[t]   = np.empty(0, dtype=np.int64)
        parts_tx_at_t[t]  = np.empty(0, dtype=np.int64)
        parts_seg_at_t[t] = np.empty(0, dtype=np.int64)

# Per-tx incident wallet tensors for attention (top-W_PER_SIDE per side, padded)
tx_w_idx = np.full((N_tx, W_TOTAL), -1, dtype=np.int64)
tx_w_seg = np.zeros((N_tx, W_TOTAL), dtype=np.int64)
tx_w_pad = np.ones((N_tx, W_TOTAL), dtype=bool)
for tx_idx in range(N_tx):
    in_w  = pick_top_wallets(list(set(incident_in.get(tx_idx,  []))), W_PER_SIDE)
    out_w = pick_top_wallets(list(set(incident_out.get(tx_idx, []))), W_PER_SIDE)
    for slot, w in enumerate(in_w):
        tx_w_idx[tx_idx, slot] = w
        tx_w_seg[tx_idx, slot] = 0
        tx_w_pad[tx_idx, slot] = False
    for slot, w in enumerate(out_w):
        idx = W_PER_SIDE + slot
        tx_w_idx[tx_idx, idx] = w
        tx_w_seg[tx_idx, idx] = 1
        tx_w_pad[tx_idx, idx] = False

print(f"  per-tx incident wallet grid: {tx_w_idx.shape}  pad_fraction={tx_w_pad.mean():.3f}")

# Standardise tx features for the neural model (fit on train only)
labelled   = (tx_label != -1)
train_mask = labelled & (tx_time <= TRAIN_END)
test_mask  = labelled & (tx_time >  TRAIN_END)
y_train = tx_label[train_mask]
y_test  = tx_label[test_mask]
test_t_arr = tx_time[test_mask]

from sklearn.preprocessing import StandardScaler
tx_scaler = StandardScaler().fit(tx_X_raw[train_mask])
tx_X_std  = tx_scaler.transform(tx_X_raw).astype(np.float32)

# Move tensors to device for CT-BNet
tx_X_std_dev   = torch.from_numpy(tx_X_std).to(DEVICE)
tx_label_dev   = torch.from_numpy(tx_label).to(DEVICE)
tx_w_idx_dev   = torch.from_numpy(tx_w_idx).to(DEVICE)
tx_w_seg_dev   = torch.from_numpy(tx_w_seg).to(DEVICE)
tx_w_pad_dev   = torch.from_numpy(tx_w_pad).to(DEVICE)

print(f"  tx_X_std (standardised, train-fit): {tx_X_std.shape}")
print(f"  train: n={int(train_mask.sum()):,}  illicit_rate={y_train.mean():.4f}")
print(f"  test:  n={int(test_mask.sum()):,}  illicit_rate={y_test.mean():.4f}")


Building chronological data structures...
  per-tx incident wallet grid: (203769, 8)  pad_fraction=0.567
  tx_X_std (standardised, train-fit): (203769, 182)
  train: n=29,894  illicit_rate=0.1158
  test:  n=16,670  illicit_rate=0.0650


/var/folders/7d/vs4kyt612wv601llgqwwx1yw0000gn/T/ipykernel_86444/2730935680.py:74: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:219.)
  tx_label_dev   = torch.from_numpy(tx_label).to(DEVICE)


## Model definitions

`TimeEncoding` — Bochner sinusoidal time encoding with learnable frequencies (continuous in `τ`, generalises to unseen test timesteps).

`GradReverse` — gradient reversal layer (Ganin & Lempitsky 2015) for the drift-adversarial term. Forward is identity; backward multiplies the gradient by `-λ_adv`.

`CTBNet` — full CT-BNet model with all four modules (memory GRU, bipartite attention, drift discriminator, fusion classifier).


In [6]:
class TimeEncoding(nn.Module):
    """Bochner sinusoidal time encoding with learnable frequencies (continuous in τ)."""
    def __init__(self, d):
        super().__init__()
        assert d % 2 == 0
        self.freq = nn.Parameter(torch.randn(d // 2) * 0.1)
    def forward(self, t):
        x = t.float().unsqueeze(-1) * self.freq
        return torch.cat([torch.cos(x), torch.sin(x)], dim=-1)


class GradReverse(torch.autograd.Function):
    """Identity in forward; flips gradient sign (× lambd) in backward.
    Used by the drift-adversarial head — the GRU/encoder fights the discriminator."""
    @staticmethod
    def forward(ctx, x, lambd):
        ctx.lambd = lambd
        return x.view_as(x)
    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.lambd * grad_output, None


class CTBNet(nn.Module):
    """Causal Temporal Bipartite Network.
    Wallet memory (GRU) + bipartite attention + drift adversary + fusion classifier."""
    def __init__(self, F_tx, n_w, d_event=64, d_mem=64, d_attn=64, d_seg=8,
                 dropout=0.3):
        super().__init__()
        self.n_w     = n_w
        self.d_event = d_event
        self.d_mem   = d_mem
        self.d_attn  = d_attn

        # Module 1: event embedding + GRU memory
        self.tx_proj  = nn.Linear(F_tx, d_event)
        self.time_enc = TimeEncoding(d_event)
        self.gru      = nn.GRUCell(d_event, d_mem)

        # Module 2: bipartite attention
        # query: [x_T (F_tx) || φ(t) (d_event)]  ->  d_attn
        # key:   [m_w (d_mem) || φ(Δt) (d_event) || seg (d_seg)]  ->  d_attn
        # value: m_w (d_mem)  ->  d_attn
        self.seg_emb = nn.Embedding(2, d_seg)
        self.q_proj  = nn.Linear(F_tx + d_event, d_attn)
        self.k_proj  = nn.Linear(d_mem + d_event + d_seg, d_attn)
        self.v_proj  = nn.Linear(d_mem, d_attn)
        self.dropout = nn.Dropout(dropout)

        # Module 3: drift discriminator (used through GradReverse)
        self.disc = nn.Sequential(
            nn.Linear(d_mem, d_mem // 2),
            nn.ReLU(),
            nn.Linear(d_mem // 2, 1),
        )

        # Module 4: fusion classifier
        self.fuse_r     = nn.Linear(d_attn, F_tx)
        self.classifier = nn.Sequential(
            nn.Linear(F_tx + d_attn + F_tx, 64),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(64, 2),
        )

    def event_embedding(self, x_tx, t):
        """e(T, t) = W_e · x_T + φ(t).  Inputs: x_tx [B, F_tx], t [B] -> [B, d_event]."""
        return self.tx_proj(x_tx) + self.time_enc(t)

    def attention_context(self, x_tx, t_T, w_mem, w_last_t, w_seg, w_pad):
        """Bipartite attention. Returns (ctx [B, d_attn], attn [B, K])."""
        B, K, _ = w_mem.shape
        d = self.d_attn
        time_T = self.time_enc(t_T)                                  # [B, d_event]
        q  = self.q_proj(torch.cat([x_tx, time_T], dim=-1))          # [B, d_attn]
        delta_t     = (t_T.unsqueeze(-1) - w_last_t).clamp(min=0)    # [B, K]
        delta_t_enc = self.time_enc(delta_t)                          # [B, K, d_event]
        seg_e       = self.seg_emb(w_seg)                             # [B, K, d_seg]
        kv_in = torch.cat([w_mem, delta_t_enc, seg_e], dim=-1)        # [B, K, d_mem+d_event+d_seg]
        k = self.k_proj(kv_in)                                        # [B, K, d_attn]
        v = self.v_proj(w_mem)                                        # [B, K, d_attn]
        # Scaled dot-product
        scores = (q.unsqueeze(1) * k).sum(dim=-1) / (d ** 0.5)        # [B, K]
        scores = scores.masked_fill(w_pad, float("-inf"))
        # All-pad rows -> 0 output (avoid NaN softmax)
        all_pad = w_pad.all(dim=-1)                                   # [B]
        safe_scores = torch.where(all_pad.unsqueeze(-1),
                                  torch.zeros_like(scores), scores)
        attn = F.softmax(safe_scores, dim=-1)
        attn = attn.masked_fill(all_pad.unsqueeze(-1), 0.0)
        ctx = (attn.unsqueeze(-1) * v).sum(dim=1)                     # [B, d_attn]
        return ctx, attn

    def classify(self, x_tx, ctx):
        """Fusion classifier: [x_T || ctx || x_T - W_r·ctx]."""
        residual = x_tx - self.fuse_r(ctx)
        h = torch.cat([x_tx, ctx, residual], dim=-1)
        return self.classifier(h)

    def disc_predict(self, mem, lambda_adv):
        """Discriminator predicts normalised timestep from memory, via grad reversal."""
        mem_rev = GradReverse.apply(mem, lambda_adv)
        return self.disc(mem_rev).squeeze(-1)


In [7]:
# Training + chronological evaluation for CT-BNet.
# - Memory is a [N_w, D_MEM] buffer kept detached. Each timestep, we read the slice
#   for active wallets, run GRU, classify with the post-update memory, backward+step,
#   and write the new memory back (detached).
# - This is K=1 TBPTT with the post-update-memory variant. The GRU is trained because
#   cls_loss flows through new_mem (which is in the autograd graph for that timestep).
# - Test time: same chronological pass, but no labels are used in updates.

def train_ctbnet(lambda_adv: float, epochs: int = EPOCHS, label: str = "CT-BNet"):
    print(f"\n=== Training {label}  (λ_adv={lambda_adv}, epochs={epochs}) ===")
    torch.manual_seed(RANDOM_SEED)
    model = CTBNet(F_tx=F_TX, n_w=N_w, d_event=D_EVENT, d_mem=D_MEM,
                   d_attn=D_ATTN, d_seg=D_SEG, dropout=DROPOUT).to(DEVICE)
    optim_ = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    cw = torch.tensor([1.0, CLASS_WEIGHT], dtype=torch.float, device=DEVICE)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"  model params: {n_params:,}")

    # Per-tx output containers (filled during the test pass of the final epoch)
    test_proba = np.zeros(N_tx, dtype=np.float32)

    def chrono_pass(train_phase: bool, do_predict: bool):
        """One full chronological pass over t = 1..N_TIME_STEPS.
        - train_phase: if True, optimize on labeled txs at t <= TRAIN_END.
        - do_predict: if True, save probas for labeled txs at t > TRAIN_END.
        """
        model.train(train_phase)
        memory_buf = torch.zeros(N_w, D_MEM, device=DEVICE)
        wallet_last_t = torch.zeros(N_w, dtype=torch.long, device=DEVICE)

        cum_cls_loss, cum_adv_loss, cum_n = 0.0, 0.0, 0

        for t in range(1, N_TIME_STEPS + 1):
            txs = txs_at_t[t]
            if txs.size == 0:
                continue
            parts_w  = parts_w_at_t[t]
            parts_tx = parts_tx_at_t[t]
            if parts_w.size == 0:
                continue

            # ── Aggregate event embeddings per active wallet at t ─────────
            tx_t = torch.from_numpy(txs).to(DEVICE)
            x_tx = tx_X_std_dev[tx_t]                                    # [N_tx_t, F_tx]
            t_T  = torch.full((tx_t.numel(),), t, dtype=torch.long, device=DEVICE)
            events = model.event_embedding(x_tx, t_T)                    # [N_tx_t, D_EVENT]

            # Map global tx idx -> local position in tx_t (for participation lookup)
            tx_to_local = {int(g): i for i, g in enumerate(txs.tolist())}
            parts_tx_local = np.fromiter((tx_to_local[int(x)] for x in parts_tx),
                                         dtype=np.int64, count=parts_tx.size)
            event_per_pair = events[torch.from_numpy(parts_tx_local).to(DEVICE)]   # [N_pairs, D_EVENT]

            # Active wallets at t (in stable order)
            unique_w_at_t, inverse_idx = np.unique(parts_w, return_inverse=True)
            unique_w_dev = torch.from_numpy(unique_w_at_t).to(DEVICE)
            inv_idx_dev  = torch.from_numpy(inverse_idx).to(DEVICE)
            n_active_w   = unique_w_at_t.shape[0]

            # Scatter-mean: aggregate event embeddings per active wallet
            agg_event_sum = torch.zeros(n_active_w, D_EVENT, device=DEVICE,
                                        dtype=event_per_pair.dtype)
            agg_event_sum.scatter_add_(0,
                inv_idx_dev.unsqueeze(-1).expand(-1, D_EVENT),
                event_per_pair)
            agg_event_count = torch.zeros(n_active_w, device=DEVICE)
            agg_event_count.scatter_add_(0, inv_idx_dev,
                torch.ones_like(inv_idx_dev, dtype=torch.float))
            agg_event = agg_event_sum / agg_event_count.unsqueeze(-1).clamp(min=1)

            # ── GRU update: m_w(t) = GRU(m_w(t⁻), agg_event_w) ─────────────
            prev_mem = memory_buf[unique_w_dev]                          # detached
            new_mem  = model.gru(agg_event, prev_mem)                    # [n_active_w, D_MEM] (in graph)

            # ── Bipartite attention + classify (using post-update memory) ──
            w_idx_safe = tx_w_idx_dev[tx_t].clamp(min=0)                 # [B, K] global wallet idx (-1 -> 0)
            w_pad      = tx_w_pad_dev[tx_t]                              # [B, K]
            w_seg      = tx_w_seg_dev[tx_t]                              # [B, K]
            # Map global wallet idx -> local position in unique_w_at_t
            global_to_local = torch.full((N_w,), -1, dtype=torch.long, device=DEVICE)
            global_to_local[unique_w_dev] = torch.arange(n_active_w, device=DEVICE)
            w_local_idx = global_to_local[w_idx_safe]                    # [B, K]; -1 if wallet not active at t
            # All incident wallets at t SHOULD be active at t (they're connected to a tx at t).
            # Defensive: any -1 -> use 0 and mark as pad.
            extra_pad   = (w_local_idx < 0) | w_pad
            w_local_idx_safe = w_local_idx.clamp(min=0)
            w_mem_attn  = new_mem[w_local_idx_safe]                       # [B, K, D_MEM]
            w_mem_attn  = w_mem_attn.masked_fill(extra_pad.unsqueeze(-1), 0.0)
            # Last-active timestamps for incident wallets (read pre-update)
            w_last      = wallet_last_t[w_idx_safe]                       # [B, K]
            ctx, _      = model.attention_context(x_tx, t_T, w_mem_attn,
                                                  w_last, w_seg, extra_pad)

            logits = model.classify(x_tx, ctx)                            # [B, 2]

            # ── Loss / metric routing ─────────────────────────────────────
            labels_t = tx_label_dev[tx_t]                                 # [B]
            if t <= TRAIN_END:
                m_train = (labels_t != -1)
                if m_train.any() and train_phase:
                    cls_loss = F.cross_entropy(logits[m_train], labels_t[m_train], weight=cw)
                    if lambda_adv > 0:
                        t_target = torch.full((n_active_w,), t / float(N_TIME_STEPS),
                                              device=DEVICE)
                        t_pred   = model.disc_predict(new_mem, lambda_adv)
                        adv_loss = F.mse_loss(t_pred, t_target)
                    else:
                        adv_loss = torch.zeros((), device=DEVICE)
                    total = cls_loss + adv_loss
                    optim_.zero_grad()
                    total.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    optim_.step()
                    cum_cls_loss += float(cls_loss.detach()) * int(m_train.sum())
                    cum_adv_loss += float(adv_loss.detach()) if lambda_adv > 0 else 0.0
                    cum_n        += int(m_train.sum())
            elif do_predict:
                # Test-set predictions; no gradient update
                with torch.no_grad():
                    proba = logits.softmax(-1)[:, 1]
                test_proba[txs] = proba.cpu().numpy()

            # ── Write new memory back to buffer (detached) ────────────────
            with torch.no_grad():
                memory_buf[unique_w_dev] = new_mem.detach()
                wallet_last_t[unique_w_dev] = t

        return cum_cls_loss / max(cum_n, 1), cum_adv_loss / max(cum_n, 1)

    # Train epochs (no test predictions yet — we only score on the final pass)
    for ep in range(1, epochs + 1):
        t0 = time.time()
        cls_l, adv_l = chrono_pass(train_phase=True, do_predict=False)
        ext = f"  adv={adv_l:.4f}" if lambda_adv > 0 else ""
        print(f"  epoch {ep:2d}  cls_loss={cls_l:.4f}{ext}  ({time.time()-t0:.1f}s)")

    # Final eval pass — compute test predictions
    print("  running final chronological eval...")
    chrono_pass(train_phase=False, do_predict=True)
    return test_proba


## Run all five ablations

Three RF ablations (A, B, C) on the same train/test split as Phase 1, then two CT-BNet variants (D = no adversarial, E = with adversarial). All evaluated on the same test set (`t ≥ 35`).


In [8]:
def fit_rf_and_eval(X_train, X_test, name):
    t0 = time.time()
    rf = RandomForestClassifier(n_estimators=200, class_weight="balanced",
                                n_jobs=-1, random_state=RANDOM_SEED)
    rf.fit(X_train, y_train)
    yp  = rf.predict(X_test)
    ypr = rf.predict_proba(X_test)[:, 1]
    f1   = f1_score(y_test, yp, pos_label=1, zero_division=0)
    auc  = roc_auc_score(y_test, ypr)
    prauc= average_precision_score(y_test, ypr)
    print(f"  [{name:30s}]  F1={f1:.4f}  AUC={auc:.4f}  PR-AUC={prauc:.4f}  ({time.time()-t0:.1f}s)")
    return rf, yp, ypr, f1, auc, prauc

# Ablation A
print("=== Ablation A: RF[raw 182] ===")
X_A_train = tx_X_raw[train_mask]; X_A_test = tx_X_raw[test_mask]
rf_A, yp_A, ypr_A, f1_A, auc_A, prauc_A = fit_rf_and_eval(X_A_train, X_A_test, "A: raw 182")

# Ablation B
print("\n=== Ablation B: RF[raw + 103 traj] ===")
X_B_train = np.concatenate([tx_X_raw[train_mask], traj_X[train_mask]], axis=1)
X_B_test  = np.concatenate([tx_X_raw[test_mask],  traj_X[test_mask]],  axis=1)
rf_B, yp_B, ypr_B, f1_B, auc_B, prauc_B = fit_rf_and_eval(X_B_train, X_B_test, "B: raw + traj")

# Ablation C
print("\n=== Ablation C: RF[raw + traj + 7 L1] ===")
X_C_train = np.concatenate([X_B_train, L1_X[train_mask]], axis=1)
X_C_test  = np.concatenate([X_B_test,  L1_X[test_mask]],  axis=1)
rf_C, yp_C, ypr_C, f1_C, auc_C, prauc_C = fit_rf_and_eval(X_C_train, X_C_test, "C: raw + traj + L1")


=== Ablation A: RF[raw 182] ===
  [A: raw 182                    ]  F1=0.8044  AUC=0.9269  PR-AUC=0.7927  (2.0s)

=== Ablation B: RF[raw + 103 traj] ===
  [B: raw + traj                 ]  F1=0.8098  AUC=0.9333  PR-AUC=0.8097  (1.9s)

=== Ablation C: RF[raw + traj + 7 L1] ===
  [C: raw + traj + L1            ]  F1=0.8094  AUC=0.9361  PR-AUC=0.8093  (2.1s)


In [9]:
# Ablation D: CT-BNet without adversarial term (λ_adv = 0)
test_proba_D = train_ctbnet(lambda_adv=0.0, label="Ablation D (CT-BNet, no adv)")
yp_D  = (test_proba_D[test_mask] >= 0.5).astype(np.int64)
ypr_D = test_proba_D[test_mask]
f1_D    = f1_score(y_test, yp_D, pos_label=1, zero_division=0)
auc_D   = roc_auc_score(y_test, ypr_D) if len(np.unique(y_test)) > 1 else float("nan")
prauc_D = average_precision_score(y_test, ypr_D)
print(f"  [D: CT-BNet (no adv)]  F1={f1_D:.4f}  AUC={auc_D:.4f}  PR-AUC={prauc_D:.4f}")

# Ablation E: full CT-BNet (λ_adv = LAMBDA_ADV_E)
test_proba_E = train_ctbnet(lambda_adv=LAMBDA_ADV_E, label=f"Ablation E (CT-BNet, λ_adv={LAMBDA_ADV_E})")
yp_E  = (test_proba_E[test_mask] >= 0.5).astype(np.int64)
ypr_E = test_proba_E[test_mask]
f1_E    = f1_score(y_test, yp_E, pos_label=1, zero_division=0)
auc_E   = roc_auc_score(y_test, ypr_E) if len(np.unique(y_test)) > 1 else float("nan")
prauc_E = average_precision_score(y_test, ypr_E)
print(f"  [E: CT-BNet (full)]  F1={f1_E:.4f}  AUC={auc_E:.4f}  PR-AUC={prauc_E:.4f}")



=== Training Ablation D (CT-BNet, no adv)  (λ_adv=0.0, epochs=8) ===
  model params: 106,985
  epoch  1  cls_loss=0.5121  (3.1s)
  epoch  2  cls_loss=0.2654  (2.9s)
  epoch  3  cls_loss=0.2770  (2.7s)
  epoch  4  cls_loss=0.1850  (2.8s)
  epoch  5  cls_loss=0.1812  (2.8s)
  epoch  6  cls_loss=0.1705  (2.6s)
  epoch  7  cls_loss=0.1888  (2.8s)
  epoch  8  cls_loss=0.1629  (2.8s)
  running final chronological eval...
  [D: CT-BNet (no adv)]  F1=0.5759  AUC=0.9160  PR-AUC=0.6147

=== Training Ablation E (CT-BNet, λ_adv=0.1)  (λ_adv=0.1, epochs=8) ===
  model params: 106,985
  epoch  1  cls_loss=0.5143  adv=0.0000  (2.8s)
  epoch  2  cls_loss=0.2681  adv=0.0001  (2.6s)
  epoch  3  cls_loss=0.2775  adv=0.0000  (2.7s)
  epoch  4  cls_loss=0.1919  adv=0.0000  (2.9s)
  epoch  5  cls_loss=0.1673  adv=0.0001  (2.7s)
  epoch  6  cls_loss=0.1762  adv=0.0001  (2.7s)
  epoch  7  cls_loss=0.1635  adv=0.0001  (2.7s)
  epoch  8  cls_loss=0.1514  adv=0.0001  (2.8s)
  running final chronological eval...

In [10]:
# === Per-timestep test F1 across all 5 ablations ===
print("=" * 90)
print("Per-timestep test F1 (illicit class)")
print("=" * 90)
print(f"{'t':>3}  {'n':>5}  {'illicit':>7}  "
      f"{'A':>7}  {'B':>7}  {'C':>7}  {'D':>7}  {'E':>7}  {'D-C':>7}  {'E-D':>7}")
for t in range(TRAIN_END + 1, N_TIME_STEPS + 1):
    m = (test_t_arr == t)
    if not m.any(): continue
    yt = y_test[m]
    if yt.sum() == 0:
        print(f"{t:>3}  {int(m.sum()):>5}  {int(yt.sum()):>7}  "
              + "  ".join(f"{'NaN':>7}" for _ in range(7)))
        continue
    f1s = {}
    for name, yp in [("A", yp_A), ("B", yp_B), ("C", yp_C), ("D", yp_D), ("E", yp_E)]:
        f1s[name] = f1_score(yt, yp[m], pos_label=1, zero_division=0)
    print(f"{t:>3}  {int(m.sum()):>5}  {int(yt.sum()):>7}  "
          f"{f1s['A']:>7.4f}  {f1s['B']:>7.4f}  {f1s['C']:>7.4f}  "
          f"{f1s['D']:>7.4f}  {f1s['E']:>7.4f}  "
          f"{f1s['D']-f1s['C']:>+7.3f}  {f1s['E']-f1s['D']:>+7.3f}")

# === Summary ===
print("\n" + "=" * 90)
print(f"{'Model':40s}  {'F1':>8s}  {'AUC':>8s}  {'PR-AUC':>8s}")
print("=" * 90)
results = {
    "A: RF[raw 182]":                      (f1_A, auc_A, prauc_A),
    "B: RF[raw + 103 traj]":               (f1_B, auc_B, prauc_B),
    "C: RF[raw + traj + 7 L1]":            (f1_C, auc_C, prauc_C),
    "D: CT-BNet (no adv)":                 (f1_D, auc_D, prauc_D),
    f"E: CT-BNet (λ_adv={LAMBDA_ADV_E})":  (f1_E, auc_E, prauc_E),
}
for name, (f1, auc, prauc) in sorted(results.items(), key=lambda kv: -kv[1][0]):
    print(f"  {name:38s}  {f1:>8.4f}  {auc:>8.4f}  {prauc:>8.4f}")

print()
print(f"Δ B vs A (Phase-1 traj signal):                    F1={f1_B-f1_A:+.4f}  PR-AUC={prauc_B-prauc_A:+.4f}")
print(f"Δ C vs B (Phase-1c bipartite-structural signal):   F1={f1_C-f1_B:+.4f}  PR-AUC={prauc_C-prauc_B:+.4f}")
print(f"Δ D vs C (learned vs hand-engineered temporal):    F1={f1_D-f1_C:+.4f}  PR-AUC={prauc_D-prauc_C:+.4f}")
print(f"Δ E vs D (drift-adversarial regularisation):       F1={f1_E-f1_D:+.4f}  PR-AUC={prauc_E-prauc_D:+.4f}")
print(f"Δ E vs C (full CT-BNet vs strongest RF):           F1={f1_E-f1_C:+.4f}  PR-AUC={prauc_E-prauc_C:+.4f}")
print(f"Δ E vs A (full system vs raw RF):                  F1={f1_E-f1_A:+.4f}  PR-AUC={prauc_E-prauc_A:+.4f}")

print("\nNotes:")
print(f"  - Same train/test split as phase 1 / 1c: train t<= {TRAIN_END}, test t > {TRAIN_END}.")
print(f"  - Causality enforced everywhere (strict τ < t for traj features; tx.t' <= t for L1;")
print(f"    chronological memory updates with no test labels in CT-BNet).")
print(f"  - CT-BNet config: D_EVENT={D_EVENT} D_MEM={D_MEM} D_ATTN={D_ATTN} W_TOTAL={W_TOTAL}")
print(f"    epochs={EPOCHS} class_weight={CLASS_WEIGHT} lr={LR}")
print(f"  - Spec relaxation: cls reads post-GRU memory at t (T's event contributes 1/n to its")
print(f"    incident wallets' aggregated update where n = #events at t for each wallet).")


Per-timestep test F1 (illicit class)
  t      n  illicit        A        B        C        D        E      D-C      E-D
 35   1341      182   0.9805   0.9775   0.9805   0.9036   0.9045   -0.077   +0.001
 36   1708       33   0.8814   0.9000   0.9180   0.4571   0.7089   -0.461   +0.252
 37    498       40   0.7500   0.7692   0.7500   0.4938   0.7059   -0.256   +0.212
 38    756      111   0.9429   0.9479   0.9429   0.5734   0.8744   -0.369   +0.301
 39   1183       81   0.9128   0.9054   0.9272   0.5238   0.6933   -0.403   +0.170
 40   1211      112   0.7416   0.7486   0.7416   0.5612   0.6629   -0.180   +0.102
 41   1132      116   0.8900   0.9005   0.9108   0.5556   0.8208   -0.355   +0.265
 42   2154      239   0.8565   0.8599   0.8434   0.7377   0.8211   -0.106   +0.083
 43   1370       24   0.0000   0.0000   0.0000   0.0000   0.0000   +0.000   +0.000
 44   1591       24   0.0690   0.0667   0.0667   0.0303   0.0345   -0.036   +0.004
 45   1221        5   0.0000   0.0000   0.0000   0